In [1]:
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank("en")
db = DocBin().from_disk("data/ner/train_split.spacy")

bad_docs = 0
bad_spans = 0

for i, doc in enumerate(db.get_docs(nlp.vocab)):
    for ent in doc.ents:

        # invalid span length
        if ent.start >= ent.end:
            print("BAD LENGTH:", ent.text)
            bad_docs += 1
            bad_spans += 1

        # whitespace
        elif ent.text != ent.text.strip():
            print("WHITESPACE:", repr(ent.text))
            bad_docs += 1
            bad_spans += 1

        # newline contamination
        elif "\n" in ent.text:
            print("NEWLINE:", repr(ent.text))
            bad_docs += 1
            bad_spans += 1

print("Bad docs:", bad_docs)
print("Bad spans:", bad_spans)


/home/abhi/miniforge3/envs/nlp_intern/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Bad docs: 0
Bad spans: 0


In [2]:
import spacy
from spacy.tokens import DocBin
from collections import Counter

def get_label_stats(spacy_path):
    nlp = spacy.blank("en")
    db = DocBin().from_disk(spacy_path)
    docs = list(db.get_docs(nlp.vocab))

    label_counter = Counter()
    for doc in docs:
        for ent in doc.ents:
            label_counter[ent.label_] += 1

    return label_counter


In [3]:
TRAIN_PATH = "data/ner/train_split.spacy"
DEV_PATH = "data/ner/val_split.spacy"

train_labels = get_label_stats(TRAIN_PATH)
dev_labels = get_label_stats(DEV_PATH)

train_set = set(train_labels.keys())
dev_set = set(dev_labels.keys())

print("Total train labels:", len(train_set))
print("Total dev labels:", len(dev_set))

print("\nLabels only in TRAIN (missing in dev):")
print(sorted(train_set - dev_set))

print("\nLabels only in DEV (missing in train):")
print(sorted(dev_set - train_set))


Total train labels: 9
Total dev labels: 9

Labels only in TRAIN (missing in dev):
[]

Labels only in DEV (missing in train):
[]


In [4]:
train_labels

Counter({'PARTY': 1570,
         'CONTRACT_DATE': 452,
         'RESTRICTION': 91,
         'LICENSE_TERM': 55,
         'TERMINATION': 49,
         'PAYMENT_TERM': 37,
         'LIABILITY': 29,
         'IP_OWNERSHIP': 24,
         'LEGAL': 22})

In [ ]:
dev_labels

In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.training import offsets_to_biluo_tags

nlp = spacy.blank("en")

db = DocBin().from_disk("data/ner/train_split.spacy")
docs = list(db.get_docs(nlp.vocab))

bad_docs = 0

for i, doc in enumerate(docs):

    entities = [(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents]

    tags = offsets_to_biluo_tags(nlp.make_doc(doc.text), entities)

    if "-" in tags:
        print(f"🚨 Bad doc at index: {i}")
        bad_docs += 1

print("TOTAL BAD DOCS:", bad_docs)
